In [ ]:
# IMPORT LIBRARIES
import os
import time
import subprocess
import platform
import psutil
import re
import gradio as gr
from google import genai
import ollama
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown
import requests
# for loading .env file
load_dotenv(override=True)

In [ ]:
# System specs for executing c++
def get_system_specs():
    return {
        "CPU": platform.processor(),
        "Cores": psutil.cpu_count(logical=False),
        "RAM": f"{round(psutil.virtual_memory().total / (1024**3), 2)} GB",
        "OS": f"{platform.system()} {platform.release()}"
    }

print("System Specs Detected:", get_system_specs())

In [ ]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



In [ ]:
clients = {
    "openai": OpenAI(api_key=openai_api_key) if openai_api_key else None,
    "gemini": OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/") if google_api_key else None,
    "groq": OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1") if groq_api_key else None,
    "openrouter": OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1") if openrouter_api_key else None,
    "ollama": OpenAI(api_key="ollama", base_url="http://localhost:11434/v1") # Local requires no real key
}

In [ ]:
AVAILABLE_MODELS = {
    # LOCAL OLLAMA
    "Qwen 2.5 Coder 7B (Local)": {"provider": "ollama", "id": "qwen2.5-coder:7b"},
    "Llama 3.1 8B (Local)": {"provider": "ollama", "id": "llama3.1"},
    
    # FREE CLOUD APIs
    "Gemini 2.5 Flash (Google Free)": {"provider": "gemini", "id": "gemini-2.5-flash"},
    "Llama 3.3 70B (Groq Free Tier)": {"provider": "groq", "id": "llama-3.3-70b-versatile"},
    "Qwen 2.5 Coder 32B (Groq Free Tier)": {"provider": "groq", "id": "qwen-2.5-coder-32b"},
    "DeepSeek R1 (OpenRouter Free)": {"provider": "openrouter", "id": "deepseek/deepseek-r1:free"},
    
    # GROK / xAI (Uses your free monthly API credits)
    "Grok 4.1 Fast (xAI)": {"provider": "grok", "id": "grok-4.1-fast"},
    "Grok 2 Latest (xAI)": {"provider": "grok", "id": "grok-2-latest"},
    
    # PAID CLOUD APIs
    "Claude 3.5 Sonnet (Paid)": {"provider": "anthropic", "id": "claude-3-5-sonnet-latest"},
    "GPT-4o (Paid)": {"provider": "openai", "id": "gpt-4o"}
}

In [ ]:
# this block will check if your system has c++ compiler installed or not, 
# It will give you instructions to install g++ compiler on your laptop that can be used ny this project to run c++ code

message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""
# Fetch the Gemini client from our new dictionary
gemini_client = clients.get("gemini")

if gemini_client:
    try:
        response = gemini_client.chat.completions.create(
            model="gemini-2.5-flash",
            messages=[
                {"role": "user", "content": message}
            ]
        )
        display(Markdown(response.choices[0].message.content))
    except Exception as e:
        print(f"Error communicating with Gemini: {e}")
else:
    print("Error: Gemini client not initialized. Check if GEMINI_API_KEY is in your .env file.")

In [ ]:
## Running the Python code:
def run_python_baseline(python_code):
    """Saves and runs the Python code, returning output and time."""
    with open("benchmark_script.py", "w") as f:
        f.write(python_code)
    
    start = time.perf_counter()
    try:
        result = subprocess.run(["python", "benchmark_script.py"], capture_output=True, text=True, timeout=15)
        end = time.perf_counter()
        output = result.stdout if result.returncode == 0 else result.stderr
        return output, (end - start)
    except subprocess.TimeoutExpired:
        return "Python Execution Timed Out (>15s)", 15.0

#### Function for converts pyhton code to fastest c++ code
#### this code is hardcoded you can use it or just below you will found optimised code for this:

def translate_python_to_cpp(python_code, model_choice):
    """Routes the prompt to the correct API based on the dictionary."""
    
    prompt = f"""You are a Senior C++ Performance Engineer.
Translate this Python code into hyper-optimized C++20.
Target: Absolute maximum execution speed.
Rules:
1. Output ONLY valid, compilable C++ code. No markdown blocks (no ```cpp).
2. Do NOT provide explanations.
3. Use efficient memory allocation and primitive types.

Python Code:
{python_code}"""
    
    config = AVAILABLE_MODELS.get(model_choice)
    if not config: 
        return "// Error: Model configuration not found."
    
    provider = config["provider"]
    model_id = config["id"]
    
    try:
        # --- OLLAMA (Local) ---
        if provider == "ollama":
            import ollama
            response = ollama.chat(model=model_id, messages=[{'role': 'user', 'content': prompt}])
            return response['message']['content']
            
        # --- GOOGLE GEMINI ---
        elif provider == "gemini":
            from google import genai # Assuming new SDK syntax based on your code
            client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
            response = client.models.generate_content(model=model_id, contents=prompt)
            return response.text
            
        # --- GROQ ---
        elif provider == "groq":
            from groq import Groq
            client = Groq(api_key=os.getenv("GROQ_API_KEY"))
            response = client.chat.completions.create(model=model_id, messages=[{"role": "user", "content": prompt}])
            return response.choices[0].message.content
            
        # --- OPENROUTER (Using standard requests library) ---
        elif provider == "openrouter":
            headers = {
                "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
                "HTTP-Referer": "[https://github.com/your-repo](https://github.com/your-repo)", # OpenRouter expects these
                "X-Title": "Python-to-Cpp-Benchmarker",
            }
            data = {"model": model_id, "messages": [{"role": "user", "content": prompt}]}
            response = requests.post("[https://openrouter.ai/api/v1/chat/completions](https://openrouter.ai/api/v1/chat/completions)", headers=headers, json=data)
            return response.json()['choices'][0]['message']['content']
            
        # --- OPENAI ---
        elif provider == "openai":
            from openai import OpenAI
            client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            response = client.chat.completions.create(model=model_id, messages=[{"role": "user", "content": prompt}])
            return response.choices[0].message.content
            
        # --- ANTHROPIC ---
        elif provider == "anthropic":
            from anthropic import Anthropic
            client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
            response = client.messages.create(model=model_id, max_tokens=2048, messages=[{"role": "user", "content": prompt}])
            return response.content[0].text
            
        else:
            return f"// Error: Provider '{provider}' not implemented."

    except Exception as e:
        return f"// Translation API Error ({provider}): {str(e)}"

In [ ]:
system_specs = get_system_specs()

def translate_python_to_cpp(python_code, model_choice):
    """Universal Router: Directs the prompt to the correct pre-initialized client."""
    prompt = f"""
You are a world-class C++ performance engineer and competitive programming expert.

Your goal is to generate the FASTEST POSSIBLE C++ code optimized for the given system.

{system_specs}


CRITICAL OBJECTIVES:
1. Use the MOST OPTIMAL ALGORITHM (reduce time complexity if possible)
2. Prioritize execution speed over readability
3. Avoid unnecessary memory allocations
4. Use cache-friendly and low-level optimizations where useful
5. Use efficient data structures (e.g., bitsets, arrays instead of vector when faster)

STRICT COMPILATION RULES:
6. MUST include ALL required headers explicitly
7. MUST compile with: g++ -O3 -march=native -std=c++20
8. MUST use std:: correctly
9. MUST include int main()
10. MUST match Python output EXACTLY

OUTPUT RULES:
- Output ONLY raw C++ code
- NO explanations
- NO markdown

Python Code:
{python_code}
"""
    
    config = AVAILABLE_MODELS.get(model_choice)
    if not config: return "// Error: Model configuration not found."
    
    provider = config["provider"]
    model_id = config["id"]
    
    try:
        # Special Case: Anthropic (Claude doesn't support the OpenAI endpoint format)
        if provider == "anthropic":
            import anthropic
            if not anthropic_api_key: return "// Error: Anthropic API Key not set."
            client = anthropic.Anthropic(api_key=anthropic_api_key)
            res = client.messages.create(
                model=model_id,
                max_tokens=2048,
                messages=[{"role": "user", "content": prompt}]
            )
            return res.content[0].text
            
        # Everything else uses the standard OpenAI client trick
        else:
            client = clients.get(provider)
            if not client: 
                return f"// Error: API Key for {provider} is missing or client not initialized."
                
            res = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
               # temperature=0.0 # Strict determinism for coding : turns oout for big models it works fine, but for small models
            )
            return res.choices[0].message.content
            
    except Exception as e:
        return f"// Translation API Error ({provider}): {str(e)}"

In [ ]:
def extract_clean_code(raw_response):
    """Extracts code blocks from LLM response using regex."""

    # Pattern 1: Look for ```cpp or ```c++ blocks
    match = re.search(r'```(?:cpp|c\+\+)\n(.*?)\n```', raw_response, re.DOTALL)
    if match:
        return match.group(1).strip()

    # Pattern 2: Look for generic ``` blocks
    match_generic = re.search(r'```\n(.*?)\n```', raw_response, re.DOTALL)
    if match_generic:
        return match_generic.group(1).strip()

    # Pattern 3: Inline single backtick (rare but useful)
    match_inline = re.search(r'`([^`]+)`', raw_response)
    if match_inline:
        return match_inline.group(1).strip()

    # Fallback: Clean conversational fluff
    cleaned = raw_response
    cleaned = cleaned.replace("Here is the code:", "")
    cleaned = cleaned.replace("Here is the C++ code:", "")
    cleaned = cleaned.strip()

    return cleaned

In [ ]:
def generate_and_clean_cpp(python_code, model_choice):
    """Translates the code and strips away the LLM markdown/yapping."""
    raw_response = translate_python_to_cpp(python_code, model_choice)
    
    # If it's an API error, just return it so you see it in the UI
    if raw_response.startswith("// Error"):
        return raw_response
        
    clean_code = extract_clean_code(raw_response)
    return clean_code

In [ ]:
import shutil

gpp_path = shutil.which("g++")
print(gpp_path)

In [ ]:
import subprocess
subprocess.run("where g++", shell=True, capture_output=True, text=True)

In [ ]:
# os.environ["PATH"] += r";C:\msys64\mingw64\bin"

In [ ]:
def find_gpp():
    """Find g++ compiler dynamically."""
    
    # 1. Check system PATH
    gpp = shutil.which("g++")
    if gpp:
        return gpp
    
    # 2. Common Windows fallback (MSYS2)
    possible_paths = [
        r"C:\msys64\mingw64\bin\g++.exe",
        r"C:\MinGW\bin\g++.exe"
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            return path
    
    return None

In [ ]:
def run_cpp_optimized(cpp_code):
    """Compiles the C++ code with extreme flags and times the execution."""
    if cpp_code.startswith("//"): 
        return "Compilation Skipped due to translation error.", 0.0

    gpp = find_gpp()
    if not gpp:
        return " g++ compiler not found. Please install MSYS2 or MinGW and add g++ to PATH.", 0.0
        
    exe_name = "benchmark_opt.exe" if platform.system() == "Windows" else "./benchmark_opt"
    cpp_file = "benchmark_opt.cpp"
    
    with open(cpp_file, "w") as f:
        f.write(cpp_code)
        
    compile_cmd = [gpp, "-Ofast", "-march=native", cpp_file, "-o", exe_name]
    comp = subprocess.run(compile_cmd, capture_output=True, text=True)
    
    if comp.returncode != 0: 
        return f"Compilation Failed:\n{comp.stderr}", 0.0
    
    start = time.perf_counter()
    try:
        exec_cmd = [f".\\{exe_name}"] if platform.system() == "Windows" else [exe_name]
        result = subprocess.run(exec_cmd, capture_output=True, text=True, timeout=15)
        end = time.perf_counter()
        return result.stdout, round((end - start), 4)
    except subprocess.TimeoutExpired:
        return "C++ Execution Timed Out (>15s)", 15.0

In [ ]:
# Checking if the cpp execute?
cpp_code = """
#include <iostream>
using namespace std;

int main() {
    cout << "Hello from C++";
    return 0;
}
"""

output, exec_time = run_cpp_optimized(cpp_code)

print("Output:", output)
print("Time:", exec_time)

In [ ]:
output, exec_time = run_cpp_optimized(cpp_code)
print(output)

In [ ]:
# Default Python code to test (Calculating Primes is a great CPU stress test)
default_python = """import time
def count_primes(n):
    count = 0
    for i in range(2, n):
        is_prime = True
        for j in range(2, int(i**0.5) + 1):
            if i % j == 0:
                is_prime = False
                break
        if is_prime: count += 1
    return count

print(f"Primes found under 50,0000: {count_primes(500000)}")
"""

In [ ]:
# Format system specs string BEFORE calling Gradio
specs_dict = get_system_specs()
specs_md = " | ".join([f"**{k}**: {v}" for k, v in specs_dict.items()])

with gr.Blocks(theme=gr.themes.Soft()) as app: # Soft theme looks a bit cleaner!
    gr.Markdown("<h1 style='text-align: center;'>⚡ AI-Driven C++ Performance Optimizer</h1>")
    gr.Markdown(f"<h3 style='text-align: center;'>🖥️ System Specs: {specs_md}</h3>")  
    
    gr.Markdown("---") # Visual divider
    
    # ---------------------------------------------------------
    # ROW 1: THE CODE EDITORS (Side-by-Side)
    # ---------------------------------------------------------
    with gr.Row():
        # Left Side: Python Input
        with gr.Column():
            gr.Markdown("### 🐍 1. Python Baseline")
            py_input = gr.Code(label="Python Code", language="python", value=default_python, lines=18)
            # Moved dropdown to the bottom left as requested
            model_dropdown = gr.Dropdown(choices=list(AVAILABLE_MODELS.keys()), label="Select AI Model (For Translation)", value=list(AVAILABLE_MODELS.keys())[0])
            
        # Right Side: C++ Output
        with gr.Column():
            gr.Markdown("### ⚙️ 2. Optimized C++")
            # interactive=True allows you to edit the AI's code before compiling!
            cpp_code_output = gr.Code(label="C++ Code", language="cpp", lines=18, interactive=True)

    # ---------------------------------------------------------
    # ROW 2: ACTION BUTTONS (All below the code blocks)
    # ---------------------------------------------------------
    with gr.Row():
        with gr.Column():
            py_run_btn = gr.Button("▶ Run Python Code", variant="primary")
        with gr.Column():
            with gr.Row():
                transpile_btn = gr.Button("✨ Generate C++", variant="primary")
                cpp_run_btn = gr.Button("🚀 Compile & Run C++", variant="stop")

    # ---------------------------------------------------------
    # ROW 3: EXECUTION RESULTS (Side-by-Side Console Outputs)
    # ---------------------------------------------------------
    with gr.Row():
        # Left Side: Python Results
        with gr.Column():
            py_time = gr.Number(label="Python Execution Time (Seconds)")
            py_output = gr.Textbox(label="Python Console Output", lines=5)
            
        # Right Side: C++ Results
        with gr.Column():
            cpp_time = gr.Number(label="C++ Execution Time (Seconds)")
            cpp_output = gr.Textbox(label="C++ Console Output", lines=5)

    # ==========================================
    # WIRE UP THE BUTTONS
    # ==========================================
    py_run_btn.click(fn=run_python_baseline, inputs=[py_input], outputs=[py_output, py_time])
    transpile_btn.click(fn=generate_and_clean_cpp, inputs=[py_input, model_dropdown], outputs=[cpp_code_output])
    cpp_run_btn.click(fn=run_cpp_optimized, inputs=[cpp_code_output], outputs=[cpp_output, cpp_time])

if __name__ == "__main__":
    app.launch(debug=True, inbrowser=True)